In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import folium
from folium import plugins
import xgboost as xgb
import lightgbm as lgb


In [21]:
def load_ecotourism_data(filepath='/content/ecotourism_dataset.csv'):
    df = pd.read_csv(filepath)
    np.random.seed(42)
    n_samples = 5000
    df = df.sample(n=n_samples, random_state=42)
    return df
df.head()

,Site_ID,Site_Name,Latitude,Longitude,Country,Biodiversity_Index,Air_Quality_Index,Elevation_m,Vegetation_Type,Flood_Risk_Index,...,Soil_Erosion_Risk,Tourist_Capacity_Limit,Current_Tourist_Count,Accessibility_Score,Human_Activity_Index,Protected_Area_Status,Conservation_Investment_USD,Climate_Risk_Score,Vulnerability_Score,Risk_Category
0,1,EcoSite_1,24.640242,-115.005170,Costa Rica,0.952546,90.066551,1629.467951,Wetland,0.805370,...,0.342746,2646.763041,1527.014838,0.246844,0.600923,False,409754.234751,0.540977,0.589754,Medium
1,2,EcoSite_2,47.034614,-77.492918,USA,0.438691,131.251329,3636.392390,Wetland,0.425871,...,0.485898,3175.939641,3531.497528,0.830474,0.339175,True,285933.568245,0.461797,0.508166,Medium
2,3,EcoSite_3,47.497417,-65.703889,Mexico,0.220856,112.485498,1653.534776,Grassland,0.484749,...,0.568089,4202.826310,3237.797393,0.569547,0.108139,False,287162.931205,0.692882,0.730067,High
3,4,EcoSite_4,28.202205,-124.709388,Brazil,0.734043,180.539023,618.689506,Temperate_Forest,0.474932,...,0.902651,940.817226,3072.108637,0.780082,0.718803,False,335315.175231,0.659486,0.547967,Medium
4,5,EcoSite_5,23.918089,-103.319770,Costa Rica,0.407602,161.084938,2736.600938,Tropical_Forest,0.984704,...,0.555140,922.132216,3751.472664,0.803363,0.486630,True,205068.125530,0.396427,0.601713,Medium


In [22]:
n_samples = 5000
data = {
        'Site_ID': range(1, n_samples + 1),
        'Site_Name': [f'EcoSite_{i}' for i in range(1, n_samples + 1)],
        'Latitude': np.random.uniform(10, 50, n_samples),
        'Longitude': np.random.uniform(-130, -60, n_samples),
        'Country': np.random.choice(['USA', 'Canada', 'Mexico', 'Costa Rica', 'Brazil'], n_samples),
        'Biodiversity_Index': np.random.uniform(0.2, 1.0, n_samples),
        'Air_Quality_Index': np.random.uniform(20, 200, n_samples),
        'Elevation_m': np.random.uniform(0, 4000, n_samples),
        'Vegetation_Type': np.random.choice(['Tropical_Forest', 'Temperate_Forest', 'Grassland',
                                           'Desert', 'Wetland', 'Mountain'], n_samples),
        'Flood_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Drought_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Temperature_C': np.random.uniform(5, 40, n_samples),
        'Annual_Rainfall_mm': np.random.uniform(200, 3000, n_samples),
        'Soil_Type': np.random.choice(['Clay', 'Sand', 'Loam', 'Rocky', 'Volcanic'], n_samples),
        'Soil_Erosion_Risk': np.random.uniform(0, 1, n_samples),
        'Tourist_Capacity_Limit': np.random.uniform(100, 5000, n_samples),
        'Current_Tourist_Count': np.random.uniform(50, 4000, n_samples),
        'Accessibility_Score': np.random.uniform(0.1, 1.0, n_samples),
        'Human_Activity_Index': np.random.uniform(0.05, 0.95, n_samples),
        'Protected_Area_Status': np.random.choice([True, False], n_samples),
        'Conservation_Investment_USD': np.random.uniform(10000, 500000, n_samples),
        'Climate_Risk_Score': np.random.uniform(0.1, 0.9, n_samples)
    }
df = pd.DataFrame(data)
df.to_csv('eco_sites.csv', index=False)

In [23]:
n_samples = 5000
data = {
        'Site_ID': range(1, n_samples + 1),
        'Site_Name': [f'EcoSite_{i}' for i in range(1, n_samples + 1)],
        'Latitude': np.random.uniform(10, 50, n_samples),
        'Longitude': np.random.uniform(-130, -60, n_samples),
        'Country': np.random.choice(['USA', 'Canada', 'Mexico', 'Costa Rica', 'Brazil'], n_samples),
        'Biodiversity_Index': np.random.uniform(0.2, 1.0, n_samples),
        'Air_Quality_Index': np.random.uniform(20, 200, n_samples),
        'Elevation_m': np.random.uniform(0, 4000, n_samples),
        'Vegetation_Type': np.random.choice(['Tropical_Forest', 'Temperate_Forest', 'Grassland',
                                           'Desert', 'Wetland', 'Mountain'], n_samples),
        'Flood_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Drought_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Temperature_C': np.random.uniform(5, 40, n_samples),
        'Annual_Rainfall_mm': np.random.uniform(200, 3000, n_samples),
        'Soil_Type': np.random.choice(['Clay', 'Sand', 'Loam', 'Rocky', 'Volcanic'], n_samples),
        'Soil_Erosion_Risk': np.random.uniform(0, 1, n_samples),
        'Tourist_Capacity_Limit': np.random.uniform(100, 5000, n_samples),
        'Current_Tourist_Count': np.random.uniform(50, 4000, n_samples),
        'Accessibility_Score': np.random.uniform(0.1, 1.0, n_samples),
        'Human_Activity_Index': np.random.uniform(0.05, 0.95, n_samples),
        'Protected_Area_Status': np.random.choice([True, False], n_samples),
        'Conservation_Investment_USD': np.random.uniform(10000, 500000, n_samples),
        'Climate_Risk_Score': np.random.uniform(0.1, 0.9, n_samples)
    }
df = pd.DataFrame(data)
df['Vulnerability_Score'] = (
        0.25 * (1 - df['Biodiversity_Index']) +
        0.15 * (df['Air_Quality_Index'] / 200) +
        0.15 * df['Flood_Risk_Index'] +
        0.15 * df['Drought_Risk_Index'] +
        0.10 * df['Soil_Erosion_Risk'] +
        0.10 * (df['Temperature_C'] / 40) +
        0.10 * df['Climate_Risk_Score']
    )
df['Vulnerability_Score'] += np.random.normal(0, 0.05, n_samples)
df['Vulnerability_Score'] = np.clip(df['Vulnerability_Score'], 0, 1)
df['Risk_Category'] = pd.cut(
        df['Vulnerability_Score'],
        bins=[0, 0.33, 0.67, 1.0],
        labels=['Low', 'Medium', 'High']
    )
df.head()

,Site_ID,Site_Name,Latitude,Longitude,Country,Biodiversity_Index,Air_Quality_Index,Elevation_m,Vegetation_Type,Flood_Risk_Index,...,Soil_Erosion_Risk,Tourist_Capacity_Limit,Current_Tourist_Count,Accessibility_Score,Human_Activity_Index,Protected_Area_Status,Conservation_Investment_USD,Climate_Risk_Score,Vulnerability_Score,Risk_Category
0,1,EcoSite_1,23.600658,-90.271489,Canada,0.663179,74.278599,796.788084,Wetland,0.304296,...,0.935202,4493.564238,568.807045,0.567161,0.205969,False,308830.748332,0.146638,0.473915,Medium
1,2,EcoSite_2,48.188598,-83.095310,Costa Rica,0.631041,113.068306,1172.992286,Grassland,0.598936,...,0.066718,1574.906653,375.323291,0.381992,0.749237,True,388103.400425,0.738505,0.529634,Medium
2,3,EcoSite_3,38.863972,-75.338191,Costa Rica,0.719621,57.860996,2125.432896,Wetland,0.125283,...,0.164477,3940.213613,372.044110,0.929816,0.932146,False,202154.185568,0.114664,0.173136,Low
3,4,EcoSite_4,47.961507,-116.960142,Costa Rica,0.431568,108.503499,1587.196975,Desert,0.629416,...,0.127203,1165.034749,2844.452809,0.615818,0.910778,False,459231.311970,0.288800,0.463498,Medium
4,5,EcoSite_5,16.914378,-110.764886,Brazil,0.960992,122.759548,1525.884232,Wetland,0.781449,...,0.088948,821.489638,1300.819539,0.373295,0.178060,False,51888.936747,0.702569,0.339931,Medium


In [24]:
X = df.drop(['Site_ID', 'Site_Name', 'Country', 'Vegetation_Type', 'Soil_Type', 'Risk_Category', 'Vulnerability_Score'], axis=1)
y = df['Risk_Category']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [26]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = model.predict(X_test)

In [27]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-score: {f1:.4f}')

Accuracy: 0.8970
Precision: 0.8983
Recall: 0.8970
F1-score: 0.8691
